In [1]:
import pandas as pd
import urllib.request
import urllib.error
import time
from bs4 import BeautifulSoup
#import re

In [2]:
headers = {'User-Agent':'Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:142.0) Gecko/20100101 Firefox/142.0'}

In [3]:
def scrape_team_season_data(url):
    """
    Scrapes game score data for regular season games for a single NHL team and a single season.

    Args: url (str) . This is the gamelog page for one team's games for one season

    Returns: pd.DataFrame: a DataFrame containing game-level data on goals for and against, with ENGs removed.
    """

    req = urllib.request.Request(url, headers=headers)
    # Error handling
    try:
        #html = urlopen(url)
        with urllib.request.urlopen(req) as response:
            html = response.read().decode('utf-8')
    except urllib.error.HTTPError as e:
        print(f"HTTP error {e} - could not retrieve data from {url}") # return null, break, or something else
        return pd.DataFrame()
    except urllib.error.URLError as e:
        print(f"URL error {e}. The server could not be found for {url}.")
        return pd.DataFrame()
    
    # the program continues. If I return or break in the exception catch, "else" isn't needed.

    soup = BeautifulSoup(html, 'html.parser')

    # Get the team and season from the url

    parts = url.split('/')
    team_name = parts[-2]
    season = parts[-1][:4]

    game_data = []

    # Go back to earlier code for finding the regular season table. I find I can find it just fine in the html

    regular_season_table = soup.find('table', {'id': 'games'})
    
    # Find the regular season table, this time using regex

    #regular_season_table = None
    # Search the entire html string for the comment block
    #comment_regex = re.compile(f'', re.DOTALL)
    #match = comment_regex.search(html)

    #if match:
    #    comment_content = match.group(0).replace('', '')
    #    hidden_soup = BeautifulSoup(comment_content, 'html.parser')
    #    regular_season_table = hidden_soup.find('table', {'id': f'{team_name}_games'})
    # Find the comment containing the table, which is not directly in the HTML treee
    #comments = soup.find_all(string=lambda text: isinstance(text, Comment))
    #for comment in comments:
    #    if f'div_{team_name}_games' in str(comment):
    #        # Parse the comment to find the table
    #        hidden_soup = BeautifulSoup(comment, 'html.parser')
    #        regular_season_table = hidden_soup.find('table', {'id': f'{team_name}_games'})
    #        if regular_season_table:
    #            break

    if not regular_season_table:
        print(f'Could not find the regular season table for {team_name} in the {season} season.')
        return pd.DataFrame()

    # Iterate through each row in the table

    for row in regular_season_table.tbody.find_all('tr'):

        time.sleep(3)
        
        # Skip rows without data or that are headers

        if 'class' in row.attrs and 'thead' in row['class']:
            continue

        cols = row.find_all('td')
        if len(cols) < 8:
            continue

        try:
            # Extract data from the game log table
            gf = int(cols[3].text.strip())
            ga = int(cols[4].text.strip())
            w_l = cols[5].text.strip()
            ot_so = cols[6].text.strip()

            # Get box score url from column 2
            box_score_link = cols[0].find('a')['href']
            box_score_url = f'https://www.hockey-reference.com{box_score_link}'
            #print(f'Checking the box score at {box_score_url} for ENGs.')

            # Adjust goal totals for shootouts
            if w_l == 'W' and ot_so == "SO":
                gf -= 1
            elif w_l == "L" and ot_so == "SO":
                ga -= 1

            # Scrape the box scores for ENGs

            en_gf = 0
            en_ga = 0

            req_bs = urllib.request.Request(box_score_url, headers=headers)

            time.sleep(3)
                
            try:
                with urllib.request.urlopen(req_bs) as response_bs:
                    box_score_html = response_bs.read().decode('utf-8')
        
                box_score_soup = BeautifulSoup(box_score_html, 'html.parser')
            
                # Find the Scoring Summary table

                scoring_summary_table = box_score_soup.find('table', {'id': 'scoring'})

                if scoring_summary_table:
                    for scoring_row in scoring_summary_table.find_all('tr'):
                        
                        if 'class' in scoring_row.attrs and 'thead' in scoring_row['class']:
                            continue
                            
                        scoring_cols = scoring_row.find_all('td')
                        if len(scoring_cols) >= 3:
                            team_code = scoring_cols[1].text.strip()
                            goal_type = scoring_cols[2].text.strip()

                            # Check for "EN" in goal_type
                            if "EN" in goal_type:
                                if team_code == team_name:
                                    en_gf += 1
                                else:
                                    en_ga += 1
            except urllib.error.HTTPError as e:
                print(f"HTTP Error as {e} - Could not retrieve box score from {box_score_url}.")
            except urllib.error.URLError as e:
                print(f"URL Error as {e} - The server could not be found for {box_score_url}.")

            gf -= en_gf
            ga -= en_ga

            game_data.append({
                "TEAM": team_name,
                "SEASON": season, 
                "GF": gf,
                "GA": ga
                })
        
        except(ValueError, IndexError) as e:
            print(f'Skipping a row due to a data error: {e}')
            continue
    
    return pd.DataFrame(game_data)

In [17]:
# Define the urls to scrape
# We can easily add to this list

urls_to_scrape = [
    "https://www.hockey-reference.com/teams/ANA/2025_games.html",
    "https://www.hockey-reference.com/teams/BOS/2025_games.html",
    "https://www.hockey-reference.com/teams/BUF/2025_games.html",
    "https://www.hockey-reference.com/teams/CAR/2025_games.html",
    "https://www.hockey-reference.com/teams/CBJ/2025_games.html",
    "https://www.hockey-reference.com/teams/CHI/2025_games.html",
    "https://www.hockey-reference.com/teams/CGY/2025_games.html",
    "https://www.hockey-reference.com/teams/COL/2025_games.html",
    "https://www.hockey-reference.com/teams/DAL/2025_games.html",
    "https://www.hockey-reference.com/teams/DET/2025_games.html",
    "https://www.hockey-reference.com/teams/EDM/2025_games.html",
    "https://www.hockey-reference.com/teams/FLA/2025_games.html",
    "https://www.hockey-reference.com/teams/LAK/2025_games.html",
    "https://www.hockey-reference.com/teams/MIN/2025_games.html",
    "https://www.hockey-reference.com/teams/MTL/2025_games.html",
    "https://www.hockey-reference.com/teams/NJD/2025_games.html",
    "https://www.hockey-reference.com/teams/NSH/2025_games.html",
    "https://www.hockey-reference.com/teams/NYI/2025_games.html",
    "https://www.hockey-reference.com/teams/NYR/2025_games.html",
    "https://www.hockey-reference.com/teams/OTT/2025_games.html",
    "https://www.hockey-reference.com/teams/PHI/2025_games.html",
    "https://www.hockey-reference.com/teams/PIT/2025_games.html",
    "https://www.hockey-reference.com/teams/SJS/2025_games.html",
    "https://www.hockey-reference.com/teams/SEA/2025_games.html",
    "https://www.hockey-reference.com/teams/STL/2025_games.html",
    "https://www.hockey-reference.com/teams/TBL/2025_games.html",
    "https://www.hockey-reference.com/teams/TOR/2025_games.html",
    "https://www.hockey-reference.com/teams/UTA/2025_games.html",
    "https://www.hockey-reference.com/teams/VAN/2025_games.html",
    "https://www.hockey-reference.com/teams/VEG/2025_games.html",
    "https://www.hockey-reference.com/teams/WPG/2025_games.html",
    "https://www.hockey-reference.com/teams/WSH/2025_games.html",
]

# And an empty list to store data frames

all_data_frames = []


In [18]:
# Loop through urls and scrape data from each one

for url in urls_to_scrape:
    print(f"Scraping data from {url} ...")
    df = scrape_team_season_data(url)
    if not df.empty:
        all_data_frames.append(df)
    print(f"Finished scraping data from {url}.")

if all_data_frames:
    final_df = pd.concat(all_data_frames, ignore_index=True)

Scraping data from https://www.hockey-reference.com/teams/ANA/2025_games.html ...
Finished scraping data from https://www.hockey-reference.com/teams/ANA/2025_games.html.
Scraping data from https://www.hockey-reference.com/teams/BOS/2025_games.html ...
Finished scraping data from https://www.hockey-reference.com/teams/BOS/2025_games.html.
Scraping data from https://www.hockey-reference.com/teams/BUF/2025_games.html ...
Finished scraping data from https://www.hockey-reference.com/teams/BUF/2025_games.html.
Scraping data from https://www.hockey-reference.com/teams/CAR/2025_games.html ...
Finished scraping data from https://www.hockey-reference.com/teams/CAR/2025_games.html.
Scraping data from https://www.hockey-reference.com/teams/CBJ/2025_games.html ...
Finished scraping data from https://www.hockey-reference.com/teams/CBJ/2025_games.html.
Scraping data from https://www.hockey-reference.com/teams/CHI/2025_games.html ...
Finished scraping data from https://www.hockey-reference.com/teams/C

In [19]:
final_df.head

<bound method NDFrame.head of      TEAM SEASON  GF  GA
0     ANA   2025   1   0
1     ANA   2025   1   3
2     ANA   2025   5   4
3     ANA   2025   3   4
4     ANA   2025   1   2
...   ...    ...  ..  ..
2619  WSH   2025   4   4
2620  WSH   2025   0   7
2621  WSH   2025   1   4
2622  WSH   2025   2   1
2623  WSH   2025   2   5

[2624 rows x 4 columns]>

In [31]:
final_df

,TEAM,SEASON,GF,GA
0,MDA,2006,5,3
1,MDA,2006,2,2
2,MDA,2006,2,3
3,MDA,2006,4,3
4,MDA,2006,1,4
...,...,...,...,...
159,BOS,2006,3,4
160,BOS,2006,1,2
161,BOS,2006,3,4
162,BOS,2006,3,4


In [20]:
final_df.to_csv('scraping_talentdist_20242025.csv', index=False)